# Behavior Cloning Data Collection

Pipeline: HuggingFace replays → padded maps + env actions → chunked `vmap` scan → `(obs, action)` pairs.

Architecture:
1. **Convert** replay maps / moves into env grids + `(T, 2, 5)` action sequences
2. **Per batch** set `T = max` game length in that batch (capped by `TRUNCATION`)
3. **Chunked rollout** — step `FLUSH_EVERY` (10) turns at a time; filter pairs and move them off GPU/RAM; keep only game state
4. **Save only live pairs** — drop post-`is_done` frames (and passes if `INCLUDE_PASSES=False`)
5. **Loop** over the dataset in chunks of `BATCH_SIZE`, write shards → Drive `MyDrive/bc_replays`


## 1. Imports & config

**Colab data flow**
1. Runtime → **GPU** · Mount Drive
2. Clone repo → `/content/generals_ai` (so `import generals` works)
3. Write shards to local SSD `/content/bc/replays`, sync each → `MyDrive/bc_replays`
4. Re-run safely — existing Drive shards with the same name are skipped


In [42]:
# Colab only — mount Drive. Shards land at MyDrive/bc_replays.
try:
    from google.colab import drive  # type: ignore

    drive.mount("/content/drive")
except ImportError:
    pass

DRIVE_OUT = "/content/drive/MyDrive/bc_replays"
LOCAL_OUT = "/content/bc/replays"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [43]:
# Colab: clone the repo so `import generals` works.
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/wade-ary/ai-generals.git"
REPO_DIR = Path("/content/generals_ai")

def _has_generals(root: Path) -> bool:
    return (root / "generals" / "__init__.py").is_file()

if _has_generals(REPO_DIR):
    print(f"repo ok: {REPO_DIR}")
elif _has_generals(Path.cwd()) or _has_generals(Path.cwd().parent):
    print(f"repo already on path (cwd={Path.cwd()})")
else:
    print(f"cloning {REPO_URL} → {REPO_DIR} ...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    assert _has_generals(REPO_DIR), f"clone succeeded but generals/ missing under {REPO_DIR}"
    print("clone done")

# Make notebook cwd the repo root on Colab (imports + relative paths).
if _has_generals(REPO_DIR):
    import os
    os.chdir(REPO_DIR)
    print(f"cwd → {Path.cwd()}")


repo ok: /content/generals_ai
cwd → /content/generals_ai


In [44]:
from __future__ import annotations

import os
import sys
from pathlib import Path
from typing import Any

# Prefer CUDA before importing JAX (Colab: Runtime → Change runtime type → GPU).
os.environ.setdefault("JAX_PLATFORMS", "cuda,cpu")

# Resolve repo root so `import generals` works from any notebook cwd.
def _find_repo_root() -> Path:
    candidates: list[Path] = []
    # Cursor / VS Code Jupyter exposes the notebook path
    nb_file = globals().get("__vsc_ipynb_file__")
    if nb_file:
        candidates.append(Path(nb_file).resolve().parent)
    candidates.append(Path.cwd().resolve())
    candidates.append(Path.cwd().resolve() / "bc")
    candidates.append(Path.cwd().resolve().parent)
    # Common Colab clone locations
    candidates.append(Path("/content/generals_ai"))
    candidates.append(Path("/content/ai-generals"))
    candidates.append(Path("/content"))

    seen: set[Path] = set()
    for start in candidates:
        for cand in [start, *start.parents]:
            if cand in seen:
                continue
            seen.add(cand)
            if (cand / "generals" / "__init__.py").is_file():
                return cand
    raise ImportError(
        "Could not locate generals/ package. "
        "On Colab: re-run the clone cell above "
        f"(expected /content/generals_ai). cwd={Path.cwd()}"
    )


_ROOT = _find_repo_root()
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

import jax
import jax.numpy as jnp
import numpy as np
from datasets import load_dataset

from generals import create_action, create_initial_state, step, get_observation
from generals.core.game import GameInfo, GameState

print(f"generals import ok  (root={_ROOT})")

# --- CUDA / device ---
DEVICE = jax.devices()[0]
print(f"jax {jax.__version__}  backend={jax.default_backend()}  device={DEVICE}")
print(f"all devices: {jax.devices()}")
IN_COLAB = Path("/content/drive/MyDrive").exists() or Path("/content").exists()
if jax.default_backend() != "gpu":
    msg = (
        "CUDA GPU not active (JAX backend="
        f"{jax.default_backend()}). On Colab: Runtime → Change runtime type → GPU, "
        "then Runtime → Restart session and re-run from the top."
    )
    if IN_COLAB:
        raise RuntimeError(msg)
    print(f"WARNING: {msg}")
else:
    print(f"using CUDA ✓  {DEVICE}")
    # Extra GPU details (nvidia-smi when available)
    try:
        kind = getattr(DEVICE, "device_kind", None) or getattr(DEVICE, "platform", "?")
        print(f"gpu kind: {kind}  id={getattr(DEVICE, 'id', '?')}")
    except Exception:
        pass
    try:
        import subprocess
        smi = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
            check=False, capture_output=True, text=True,
        )
        if smi.returncode == 0 and smi.stdout.strip():
            for line in smi.stdout.strip().splitlines():
                print(f"nvidia-smi: {line.strip()}")
        else:
            print("nvidia-smi: unavailable")
    except FileNotFoundError:
        print("nvidia-smi: not found")

# Colab paths (from mount cell) — local SSD for fast writes, Drive for durable save.
DRIVE_OUT = Path(globals().get("DRIVE_OUT") or "/content/drive/MyDrive/bc_replays")
LOCAL_OUT = Path(globals().get("LOCAL_OUT") or "/content/bc/replays")
IN_COLAB = Path("/content/drive/MyDrive").exists()

if IN_COLAB:
    OUT_DIR = LOCAL_OUT
    SYNC_DIR = DRIVE_OUT
else:
    OUT_DIR = Path("data_noaug")
    SYNC_DIR = None

# Fixed map pad for vmap (maps in this dataset are 17–23)
PAD_TO = 23
TRUNCATION = 1001         # hard cap on per-batch T (max turns we will ever scan)
BATCH_SIZE = 512           # games per vmap batch
FLUSH_EVERY = 10          # evacuate BC pairs off GPU/RAM every this many turns
# Peak obs ≈ BATCH_SIZE * FLUSH_EVERY * 2 * 14 * 23 * 23 * 4 (not full T).
# T4 16GB → 64–128; A100 → try 128–256. Drop BATCH_SIZE if you OOM.
SEED = 0
INCLUDE_PASSES = False    # if False, BC pairs keep only real moves (pass==0)
MAX_GAMES = None          # None = all replays; set e.g. 128 to smoke-test a few batches
# Per-batch T = min(TRUNCATION, max move-list length in that batch).
# Shorter games still step until T (vmap lockstep) but post-done frames are not saved.

print(f"OUT_DIR (write): {OUT_DIR.resolve()}")
print(f"SYNC_DIR (Drive): {SYNC_DIR.resolve() if SYNC_DIR else '(local only — not on Colab)'}")
print(f"batch={BATCH_SIZE}  T_cap={TRUNCATION}  max_games={MAX_GAMES}")
_est = BATCH_SIZE * FLUSH_EVERY * 2 * 14 * PAD_TO * PAD_TO * 4
print(f"flush every {FLUSH_EVERY} turns")
print(f"est peak obs @ chunk={FLUSH_EVERY}: {_est/1e9:.2f} GB  (plus JAX scratch — lower BATCH_SIZE if OOM)")


generals import ok  (root=/content/generals_ai)
jax 0.7.2  backend=gpu  device=cuda:0
all devices: [CudaDevice(id=0)]
using CUDA ✓  cuda:0
gpu kind: NVIDIA A100-SXM4-80GB  id=0
nvidia-smi: NVIDIA A100-SXM4-80GB, 81920 MiB, 580.82.07
OUT_DIR (write): /content/bc/replays
SYNC_DIR (Drive): /content/drive/MyDrive/bc_replays
batch=512  T_cap=1001  max_games=None
flush every 10 turns
est peak obs @ chunk=10: 0.30 GB  (plus JAX scratch — lower BATCH_SIZE if OOM)


## 2. Load HuggingFace replays

In [45]:
print("Loading dataset...")
dataset = load_dataset("strakammm/generals_io_replays")
train_dataset = dataset["train"]
print(f"Replays: {len(train_dataset)}")
print(f"Columns: {train_dataset.column_names}")


Loading dataset...
Replays: 18803
Columns: ['version', 'id', 'mapWidth', 'mapHeight', 'usernames', 'stars', 'cities', 'cityArmies', 'generals', 'mountains', 'moves']


## 3. Action conversion

Dataset move: `[player_index, start_tile, end_tile, is_50%_move, turn_number]`

Env action: `[pass, row, col, direction, split]` with direction `0↑ 1↓ 2← 3→`.

Tile index → `(row, col)` via `row, col = divmod(tile, mapWidth)`.


In [46]:
PASS = np.asarray(create_action(to_pass=True), dtype=np.int32)

DELTA_TO_DIR = {
    (-1, 0): 0,  # UP
    (1, 0): 1,   # DOWN
    (0, -1): 2,  # LEFT
    (0, 1): 3,   # RIGHT
}


def tile_to_rc(tile: int, width: int) -> tuple[int, int]:
    return divmod(int(tile), int(width))


def tiles_to_direction(start_tile: int, end_tile: int, width: int) -> int:
    sr, sc = tile_to_rc(start_tile, width)
    er, ec = tile_to_rc(end_tile, width)
    key = (er - sr, ec - sc)
    if key not in DELTA_TO_DIR:
        raise ValueError(f"non-adjacent move {start_tile}->{end_tile} (delta={key})")
    return DELTA_TO_DIR[key]


def dataset_move_to_action(start_tile: int, end_tile: int, is_50: int, width: int) -> np.ndarray:
    """Convert one dataset move into env action [pass, row, col, direction, split]."""
    row, col = tile_to_rc(start_tile, width)
    direction = tiles_to_direction(start_tile, end_tile, width)
    return np.array([0, row, col, direction, int(is_50)], dtype=np.int32)


# quick sanity check
_w = 19
_a = dataset_move_to_action(20, 39, 0, _w)  # down one row
assert _a.tolist() == [0, 1, 1, 1, 0], _a
print("action conversion ok:", _a)


action conversion ok: [0 1 1 1 0]


## 4. Map + action-sequence builders

Grid encoding for `create_initial_state`:
`-2` mountain · `0` empty · `1` P0 general · `2` P1 general · `>2` city army value

Pads every map to `(PAD_TO, PAD_TO)` with mountains so a batch can be `vmap`'d.


In [47]:
def replay_to_grid(replay: dict[str, Any], pad_to: int = PAD_TO) -> np.ndarray:
    """Build a padded numeric grid from one HF replay."""
    h, w = int(replay["mapHeight"]), int(replay["mapWidth"])
    if h > pad_to or w > pad_to:
        raise ValueError(f"map {(h, w)} exceeds pad_to={pad_to}")

    grid = np.zeros((h, w), dtype=np.int32)

    for tile in replay["mountains"]:
        r, c = tile_to_rc(tile, w)
        grid[r, c] = -2

    for tile, army in zip(replay["cities"], replay["cityArmies"]):
        r, c = tile_to_rc(tile, w)
        grid[r, c] = int(army)

    for player, tile in enumerate(replay["generals"]):
        r, c = tile_to_rc(tile, w)
        grid[r, c] = player + 1  # 1 or 2

    padded = np.full((pad_to, pad_to), -2, dtype=np.int32)
    padded[:h, :w] = grid
    return padded


def replay_num_turns(replay: dict[str, Any], truncation: int = TRUNCATION) -> int:
    """How many scan steps this replay needs (last move turn + 1), capped."""
    moves = replay["moves"]
    if len(moves) == 0:
        return 1
    max_turn = max(int(m[4]) for m in moves)
    return int(min(truncation, max_turn + 1))


def replay_to_actions(replay: dict[str, Any], truncation: int = TRUNCATION) -> np.ndarray:
    """
    Build a (T, 2, 5) action sequence.

    Turns with no recorded move for a player stay as pass.
    Moves with turn_number >= truncation are dropped.
    """
    w = int(replay["mapWidth"])
    seq = np.broadcast_to(np.stack([PASS, PASS]), (truncation, 2, 5)).copy()

    for move in replay["moves"]:
        player, start, end, is_50, turn = (int(x) for x in move)
        if turn >= truncation:
            continue
        seq[turn, player] = dataset_move_to_action(start, end, is_50, w)

    return seq


# smoke test on one replay
_sample = train_dataset[0]
_grid = replay_to_grid(_sample)
_T = replay_num_turns(_sample, truncation=64)
_acts = replay_to_actions(_sample, truncation=_T)
print(f"grid {_grid.shape}, nonzero={np.count_nonzero(_grid != -2)}")
print(f"turns={_T}  actions {_acts.shape}, non-pass moves={int(np.sum(_acts[:, :, 0] == 0))}")


grid (23, 23), nonzero=295
turns=64  actions (64, 2, 5), non-pass moves=81


## 5. Batch builders — stack N replays into vmappable arrays

In [48]:
def build_batch(
    replays: list[dict[str, Any]],
    truncation: int = TRUNCATION,
    pad_to: int = PAD_TO,
) -> tuple[GameState, jnp.ndarray]:
    """
    Convert a list of replays into batched initial states + actions.

    Returns:
        initial_states: GameState pytree with leading axis N
        actions: (N, T, 2, 5) int32
    """
    grids = jax.device_put(
        jnp.stack([jnp.asarray(replay_to_grid(r, pad_to)) for r in replays]),
        DEVICE,
    )
    actions = jax.device_put(
        jnp.stack([jnp.asarray(replay_to_actions(r, truncation)) for r in replays]),
        DEVICE,
    )
    initial_states = jax.vmap(create_initial_state)(grids)
    return initial_states, actions


def sample_replays(dataset, n: int, seed: int = SEED) -> list[dict[str, Any]]:
    rng = np.random.default_rng(seed)
    idxs = rng.choice(len(dataset), size=n, replace=False)
    return [dataset[int(i)] for i in idxs]


print("build_batch ready")


build_batch ready


## 6. `scan_one_game` — rollout chunk with `lax.scan`

At each timestep:
1. Build fogged `tensor_obs` for both players via `get_observation(...).as_tensor()` → `(2, 14, H, W)`
2. Apply the joint action with `step`
3. Carry the next state; emit `(obs, action, info)`

Used on **short chunks** (`FLUSH_EVERY` turns), not the full game at once.


In [49]:
def scan_one_game(
    initial_state: GameState,
    actions: jnp.ndarray,
) -> tuple[GameState, jnp.ndarray, jnp.ndarray, GameInfo]:
    """
    Run one game for len(actions) steps (a chunk).

    Args:
        initial_state: single GameState
        actions: (T_chunk, 2, 5)

    Returns:
        final_state: GameState after the chunk
        obs:   (T_chunk, 2, 14, H, W)  — tensor obs *before* each action
        acts:  (T_chunk, 2, 5)
        infos: GameInfo with leading time axis (T_chunk, ...)
    """

    def body(state: GameState, action: jnp.ndarray):
        obs0 = get_observation(state, 0).as_tensor()
        obs1 = get_observation(state, 1).as_tensor()
        obs = jnp.stack([obs0, obs1], axis=0)  # (2, 14, H, W)
        new_state, info = step(state, action)
        return new_state, (obs, action, info)

    final_state, (obs, acts, infos) = jax.lax.scan(body, initial_state, actions)
    return final_state, obs, acts, infos


print("scan_one_game defined")


scan_one_game defined


## 7. `run_batch_chunk` — `vmap` over N games for one chunk

Shapes in / out (chunk length = `FLUSH_EVERY`, last chunk padded then trimmed):
- `states`: batched `GameState` `(N, ...)`
- `actions`: `(N, T_chunk, 2, 5)`
- returns `new_states`, `obs`, `acts`, `infos`


In [50]:
@jax.jit
def run_batch_chunk(
    states: GameState,
    actions: jnp.ndarray,
) -> tuple[GameState, jnp.ndarray, jnp.ndarray, GameInfo]:
    """Parallel chunk rollouts via vmap(scan_one_game). Returns updated states."""
    return jax.vmap(scan_one_game)(states, actions)


# Keep old name as alias for any leftover cells.
run_batch = run_batch_chunk

print("run_batch_chunk defined (jit + vmap)")


run_batch_chunk defined (jit + vmap)


## 8. Flatten chunk → supervised `(obs, action)` + metadata

Each player-timestep is one BC example. Chunks are filtered immediately and concatenated on host;
GPU only ever holds ~`FLUSH_EVERY` turns of obs.


In [51]:
def alive_mask_from_infos(
    infos: GameInfo,
    prev_done: np.ndarray | None = None,
) -> np.ndarray:
    """(N, T) True where the game was still ongoing at decision time.

    `infos.is_done[t]` is after the action at t.
    `prev_done` is per-game done flag *before* this chunk (from prior chunk).
    """
    done = np.asarray(infos.is_done)  # (N, T)
    alive = np.empty(done.shape, dtype=bool)
    if prev_done is None:
        alive[:, 0] = True
    else:
        alive[:, 0] = ~np.asarray(prev_done, dtype=bool)
    if done.shape[1] > 1:
        alive[:, 1:] = ~done[:, :-1]
    return alive


def trajectories_to_bc_pairs(
    obs: jnp.ndarray,
    acts: jnp.ndarray,
    game_ids: list[str] | np.ndarray,
    include_passes: bool = INCLUDE_PASSES,
    alive: np.ndarray | None = None,
    turn_offset: int = 0,
) -> dict[str, np.ndarray]:
    """
    Flatten a trajectory *chunk* into supervised pairs + metadata.

    Args:
        obs:      (N, T, 2, 14, H, W)
        acts:     (N, T, 2, 5)
        game_ids: length-N list/array of replay ids
        alive:    optional (N, T) bool — False = post-game filler; drop
        turn_offset: absolute turn index of chunk start (for `turn` metadata)
    """
    n, t, p = acts.shape[:3]
    assert len(game_ids) == n, f"expected {n} game_ids, got {len(game_ids)}"

    x = np.asarray(obs).reshape(n * t * p, *obs.shape[3:])
    y = np.asarray(acts).reshape(n * t * p, 5)

    game_id = np.repeat(np.asarray(game_ids, dtype=object), t * p)
    player = np.tile(np.arange(p, dtype=np.int32), n * t)
    turn = np.repeat(np.arange(t, dtype=np.int32), p)
    turn = np.tile(turn, n) + int(turn_offset)

    keep = np.ones(n * t * p, dtype=bool)
    if alive is not None:
        alive_nt = np.asarray(alive)
        assert alive_nt.shape == (n, t), f"alive shape {alive_nt.shape} != {(n, t)}"
        keep &= np.repeat(alive_nt[:, :, None], p, axis=2).reshape(-1)
    if not include_passes:
        keep &= y[:, 0] == 0

    x, y = x[keep], y[keep]
    game_id, player, turn = game_id[keep], player[keep], turn[keep]

    return {
        "obs": x,
        "actions": y,
        "game_id": game_id.astype(str),
        "player": player,
        "turn": turn,
    }


def concat_bc_parts(parts: list[dict[str, np.ndarray]]) -> dict[str, np.ndarray]:
    parts = [p for p in parts if int(p["obs"].shape[0]) > 0]
    if not parts:
        return {
            "obs": np.zeros((0, 14, PAD_TO, PAD_TO), dtype=np.int32),
            "actions": np.zeros((0, 5), dtype=np.int32),
            "game_id": np.array([], dtype=str),
            "player": np.zeros((0,), dtype=np.int32),
            "turn": np.zeros((0,), dtype=np.int32),
        }
    return {k: np.concatenate([p[k] for p in parts], axis=0) for k in parts[0]}


print("trajectories_to_bc_pairs defined")


trajectories_to_bc_pairs defined


## 9. Warmup compile (one chunk)

JIT-compile `run_batch_chunk` on `FLUSH_EVERY` turns (static chunk length).


In [52]:
# Warmup: one FLUSH_EVERY chunk so later iterations hit the compiled path.
_warmup = [train_dataset[i] for i in range(min(BATCH_SIZE, len(train_dataset)))]
_T = max(replay_num_turns(r) for r in _warmup)
_ws, _wa = build_batch(_warmup, truncation=_T, pad_to=PAD_TO)
_chunk_t = min(FLUSH_EVERY, _T)
# Pad to FLUSH_EVERY for a stable compile shape.
if _chunk_t < FLUSH_EVERY:
    _pad = jnp.broadcast_to(
        jax.device_put(jnp.asarray(PASS), DEVICE),
        (_wa.shape[0], FLUSH_EVERY - _chunk_t, 2, 5),
    )
    _wa_chunk = jnp.concatenate([_wa[:, :_chunk_t], _pad], axis=1)
else:
    _wa_chunk = _wa[:, :FLUSH_EVERY]
_ = run_batch_chunk(_ws, _wa_chunk)
del _
print(f"warmup done  batch={len(_warmup)}  chunk={FLUSH_EVERY}  pad={PAD_TO}")


warmup done  batch=512  chunk=10  pad=23


## 10. Single-game sanity check (optional)

Replay game 0 through `scan_one_game` and confirm every non-pass action was legal
under fogged ownership at decision time.


In [53]:
from generals.core.action import compute_valid_move_mask

def count_illegal_moves(replay: dict[str, Any], truncation: int = TRUNCATION) -> dict[str, int]:
    grid = jnp.asarray(replay_to_grid(replay))
    actions = jnp.asarray(replay_to_actions(replay, truncation))
    state = create_initial_state(grid)

    illegal = 0
    applied = 0
    for t in range(truncation):
        action = actions[t]
        for p in range(2):
            a = action[p]
            if int(a[0]) == 1:
                continue
            applied += 1
            o = get_observation(state, int(p))
            mask = compute_valid_move_mask(o.armies, o.owned_cells, o.mountains)
            if not bool(mask[int(a[1]), int(a[2]), int(a[3])]):
                illegal += 1
        state, info = step(state, action)
        if bool(info.is_done):
            # remaining actions are pads / post-game
            break

    return {
        "applied": applied,
        "illegal": illegal,
        "winner": int(info.winner),
        "end_time": int(info.time),
    }


stats = count_illegal_moves(train_dataset[0], truncation=TRUNCATION)
print(stats)
assert stats["illegal"] == 0, stats


{'applied': 428, 'illegal': 0, 'winner': 0, 'end_time': 241}


## 11. Collect **all** games — chunked `vmap` loop

Walks the HuggingFace split in chunks of `BATCH_SIZE`:

1. `T = max(replay_num_turns)` over real games in the shard
2. Roll out in steps of `FLUSH_EVERY` (10) turns → filter pairs → **drop chunk from RAM**
3. Carry only game state (+ done flags) between chunks
4. Concatenate host pairs → write `.npz` → sync Drive

Set `MAX_GAMES` in config to limit how many replays to process.


In [ ]:
import gc
import shutil
import time
from pathlib import Path

# Prefer paths from config cell; fall back for a fresh kernel.
OUT_DIR = Path(globals().get("OUT_DIR") or "data_noaug")
SYNC_DIR = globals().get("SYNC_DIR")
FLUSH_EVERY = int(globals().get("FLUSH_EVERY") or 10)
if SYNC_DIR is not None:
    SYNC_DIR = Path(SYNC_DIR)

OUT_DIR.mkdir(parents=True, exist_ok=True)
if SYNC_DIR is not None:
    SYNC_DIR.mkdir(parents=True, exist_ok=True)

n_total = len(train_dataset) if MAX_GAMES is None else min(int(MAX_GAMES), len(train_dataset))
n_shards = (n_total + BATCH_SIZE - 1) // BATCH_SIZE
print(f"collecting {n_total}/{len(train_dataset)} games  |  batch={BATCH_SIZE}  shards={n_shards}  T_cap={TRUNCATION}")
print(f"write: {OUT_DIR.resolve()}")
print(f"Drive sync: {SYNC_DIR.resolve() if SYNC_DIR else '(disabled)'}")
print(f"chunked rollout: flush every {FLUSH_EVERY} turns (pairs leave RAM; state stays)")
print("progress summary every 10 batches")

_pass_dev = jax.device_put(jnp.asarray(PASS), DEVICE)

shard_paths: list[Path] = []
total_pairs = 0
total_games = 0
t0 = time.perf_counter()

for shard_idx, start in enumerate(range(0, n_total, BATCH_SIZE)):
    end = min(start + BATCH_SIZE, n_total)
    n_real = end - start
    batch_num = shard_idx + 1

    replays = [train_dataset[i] for i in range(start, end)]
    T_batch = max(replay_num_turns(r) for r in replays)
    out_path = OUT_DIR / f"bc_shard_{shard_idx:04d}_n{n_real}_t{T_batch}.npz"
    drive_path = (SYNC_DIR / out_path.name) if SYNC_DIR is not None else None

    if drive_path is not None and drive_path.exists() and drive_path.stat().st_size > 0:
        print(f"shard {batch_num:4d}/{n_shards}  skip (exists on Drive)  → {drive_path.name}")
        shard_paths.append(drive_path)
        total_games += n_real
    else:
        if n_real < BATCH_SIZE:
            replays = replays + [replays[0]] * (BATCH_SIZE - n_real)

        game_ids_all = [str(r["id"]) for r in replays]
        states, actions = build_batch(replays, truncation=T_batch, pad_to=PAD_TO)
        game_ids = game_ids_all[:n_real]

        bc_parts: list[dict[str, np.ndarray]] = []
        prev_done = np.zeros((BATCH_SIZE,), dtype=bool)
        last_winners = None
        n_chunks = 0

        for t_off in range(0, T_batch, FLUSH_EVERY):
            t_len = min(FLUSH_EVERY, T_batch - t_off)
            chunk = actions[:, t_off : t_off + t_len]
            if t_len < FLUSH_EVERY:
                pad = jnp.broadcast_to(
                    _pass_dev, (chunk.shape[0], FLUSH_EVERY - t_len, 2, 5)
                )
                chunk = jnp.concatenate([chunk, pad], axis=1)

            states, obs, acts, infos = run_batch_chunk(states, chunk)

            # Trim pad turns; keep only real games for host pairs.
            obs = obs[:n_real, :t_len]
            acts = acts[:n_real, :t_len]
            infos_real = jax.tree.map(lambda x: x[:n_real, :t_len], infos)

            alive = alive_mask_from_infos(infos_real, prev_done=prev_done[:n_real])
            part = trajectories_to_bc_pairs(
                obs, acts, game_ids, alive=alive, turn_offset=t_off
            )
            bc_parts.append(part)

            prev_done = np.asarray(infos.is_done[:, t_len - 1])
            last_winners = np.asarray(infos.winner[:n_real, t_len - 1])
            n_chunks += 1

            # Drop chunk arrays now; only `states` / flags carry forward.
            del obs, acts, infos, infos_real, alive, part, chunk
            gc.collect()

            if bool(prev_done[:n_real].all()):
                break

        bc = concat_bc_parts(bc_parts)
        del bc_parts, states, actions
        gc.collect()

        tmp_path = out_path.with_name(out_path.stem + ".tmp.npz")
        np.savez_compressed(
            tmp_path,
            obs=bc["obs"],
            actions=bc["actions"],
            game_id=bc["game_id"],
            player=bc["player"],
            turn=bc["turn"],
        )
        tmp_path.replace(out_path)

        if drive_path is not None:
            shutil.copy2(out_path, drive_path)

        sync_note = "  + Drive" if drive_path is not None else ""
        w = last_winners if last_winners is not None else np.full(n_real, -1)
        print(
            f"shard {batch_num:4d}/{n_shards}  games[{start}:{end}]  T={T_batch}  "
            f"chunks={n_chunks}  pairs={bc['obs'].shape[0]:,}  "
            f"P0={int(np.sum(w == 0))} P1={int(np.sum(w == 1))} "
            f"unfin={int(np.sum(w < 0))}  → {out_path.name}{sync_note}"
        )

        shard_paths.append(drive_path if drive_path is not None else out_path)
        total_pairs += int(bc["obs"].shape[0])
        total_games += n_real
        del bc

    if batch_num % 10 == 0 or batch_num == n_shards:
        elapsed = time.perf_counter() - t0
        pct = 100.0 * total_games / max(n_total, 1)
        rate = total_games / elapsed if elapsed > 0 else 0.0
        remaining = n_total - total_games
        eta_s = remaining / rate if rate > 0 else float("inf")
        print(
            f"PROGRESS  {batch_num}/{n_shards} batches  ({pct:.1f}%)  "
            f"games={total_games:,}/{n_total:,}  pairs={total_pairs:,}  "
            f"elapsed={elapsed/60:.1f}m  eta={eta_s/60:.1f}m  "
            f"{rate:.1f} games/s",
            flush=True,
        )

print(f"\ndone. games={total_games:,}  pairs={total_pairs:,}  shards={len(shard_paths)}")
print(f"local: {OUT_DIR.resolve()}")
if SYNC_DIR is not None:
    print(f"Drive: {SYNC_DIR.resolve()}  ({len(list(SYNC_DIR.glob('bc_shard_*.npz')))} shards)")
if shard_paths:
    print(f"first: {shard_paths[0]}")
    print(f"last:  {shard_paths[-1]}")


collecting 18803/18803 games  |  batch=512  shards=37  T_cap=1001
write: /content/bc/replays
Drive sync: /content/drive/MyDrive/bc_replays
chunked rollout: flush every 10 turns (pairs leave RAM; state stays)
progress summary every 10 batches
shard    1/37  games[0:512]  T=995  chunks=100  pairs=273,411  P0=230 P1=235 unfin=47  → bc_shard_0000_n512_t995.npz  + Drive
shard    2/37  games[512:1024]  T=997  chunks=100  pairs=276,613  P0=248 P1=240 unfin=24  → bc_shard_0001_n512_t997.npz  + Drive
shard    3/37  games[1024:1536]  T=939  chunks=94  pairs=289,802  P0=230 P1=238 unfin=44  → bc_shard_0002_n512_t939.npz  + Drive
shard    4/37  games[1536:2048]  T=998  chunks=100  pairs=223,495  P0=241 P1=242 unfin=29  → bc_shard_0003_n512_t998.npz  + Drive
shard    5/37  games[2048:2560]  T=994  chunks=100  pairs=228,133  P0=234 P1=246 unfin=32  → bc_shard_0004_n512_t994.npz  + Drive
shard    6/37  games[2560:3072]  T=975  chunks=98  pairs=248,823  P0=235 P1=251 unfin=26  → bc_shard_0005_n512_t97